# 02 — Accuracy Benchmarks (UAS / LAS)

Run both parsers on full EWT + SynTagRus test sets using gold tokenization.

In [ ]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import pandas as pd
from tqdm import tqdm

from src.data import load_sentences
from src.parsers import SpacyParser, StanzaParser
from src.metrics import uas, las, Gold, Prediction

EN = load_sentences(Path("../data/en_ewt_test.conllu"))
RU = load_sentences(Path("../data/ru_syntagrus_test.conllu"))
print(f"EN: {len(EN)} sents | RU: {len(RU)} sents")

In [ ]:
CHUNK = 100

def parse_dataset(parser, sents, label):
    toks = [s.tokens for s in sents]
    parses = []
    for i in tqdm(range(0, len(toks), CHUNK), desc=label):
        parses.extend(parser.parse(toks[i:i+CHUNK]))
    preds = [Prediction(p.heads, p.deprels) for p in parses]
    golds = [Gold(s.heads, s.deprels) for s in sents]
    return uas(preds, golds), las(preds, golds), preds

rows = []
parser_configs = [
    ("en", EN, SpacyParser("en_core_web_trf")),
    ("en", EN, StanzaParser("en")),
    ("ru", RU, SpacyParser("ru_core_news_lg")),
    ("ru", RU, StanzaParser("ru")),
]

all_preds = {}
for lang, sents, parser in parser_configs:
    u, l, preds = parse_dataset(parser, sents, f"{lang}-{parser.name}")
    all_preds[(lang, parser.name)] = preds
    rows.append({"lang": lang, "parser": parser.name,
                 "family": "transition" if "spacy" in parser.name else "graph",
                 "uas": round(u, 4), "las": round(l, 4),
                 "n_sentences": len(sents)})
    print(f"{lang} {parser.name:35s}  UAS={u:.4f}  LAS={l:.4f}")

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("../results/accuracy.csv", index=False)
print("Saved results/accuracy.csv")
df

## What we learned
- See CSV for headline numbers
- Expected: graph-based (Stanza) higher LAS on Russian
- Ready for speed/memory benchmarks (notebook 03)